> **Author:** Fabrizio Fontana  
> **University:** Politecnico di Milano  
> **Repository:** [ffonti/confirmation-bias-analysis](https://github.com/ffonti/confirmation-bias-analysis)  
> **Supervisor:** Prof. Cinzia Cappiello  
> **Co-supervisor:** Dott. Mattia Sabella

# **GPT Judge Evaluation**
This notebook measures *Confirmation Bias* using LLM-as-a-judge (with gpt-4o), instructing it to rate the coherence on a scale of 0-10.

In [1]:
import sys
import os
import pandas as pd

# Add the parent directory to the system path to allow imports from the src directory
sys.path.append(os.path.abspath('..'))

from src.evaluators.gpt_judge import compute_gpt_metrics

/Users/fabrizio/Desktop/Tesi/confirmation-bias-analysis/src/config.py:17: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## **Data Loading**
Upload one of the logs generated in the pipeline located in the `data/interim/` folder.

In [2]:
# Define the path to the generated results (JSONL format)
file_path = "../data/interim/3_fever/qwen2.5/3_fever_qwen2.5_results.jsonl"
# file_path = "../data/interim/4_truthfulqa/qwen2.5/4_truthfulqa_qwen2.5_results.jsonl"
# file_path = "../data/interim/5_mmlu_pro/qwen2.5/5_mmlu_pro_qwen2.5_results.jsonl"

try:
    # Load the generated results into a DataFrame
    df_results = pd.read_json(file_path, lines=True)
    print(f"Loaded {len(df_results)} samples.")
except FileNotFoundError:
    print(f"File not found at the requested path ({file_path}).")

Loaded 500 samples.


## **Evaluation**
Compute the evaluation metrics by sending the generated results to GPT, and extract the final score (parsed list).

In [3]:
# Extract filename from the input path to build the progressive save path
import os
base_name = os.path.basename(file_path).replace("_results.jsonl", "")
output_dir = os.path.dirname(file_path)
output_file = os.path.join(output_dir, f"{base_name}_gpt.csv")

# Compute the GPT metrics (will save progressively to output_file and resume if present)
df_evaluated = compute_gpt_metrics(df_results, output_file=output_file, model="gpt-4o", sleep_time=1.0)

Starting GPT evaluation as judge for 500 samples...
Error during agreement evaluation: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


KeyboardInterrupt: 

## **Summary**
Compute summary statistics on the evaluated results, and save the final DataFrame to `data/processed/` for later use.

In [ ]:
# Aggregation of the average scores for each prompt type
mean_neutral = df_evaluated["score_neutral"].mean()
mean_leading = df_evaluated["score_leading"].mean()
mean_contra = df_evaluated["score_contradictory"].mean()

print("Results of the GPT Judge Evaluation:")
print(f"Average Support (Neutral Prompt):        {mean_neutral:.2f} / 10")
print(f"Average Support (Leading Prompt):        {mean_leading:.2f} / 10")
print(f"Average Support (Contradictory Prompt):  {mean_contra:.2f} / 10")

# If the Leading prompt increases adherence and the Contradictory prompt decreases it significantly, 
# the numerical discrepancy indicates that the tested model suffers from confirmation bias.

display(df_evaluated[["sample", "model", "claim", "score_neutral", "score_leading", "score_contradictory"]].head(10))

## **Exporting**
Export the results to a CSV file for later use in the final analysis notebook.

In [ ]:
# Extract filename from the input path
base_name = os.path.basename(file_path).replace("_results.jsonl", "")
# Save the evaluated DataFrame to the exact same folder as the input file
output_dir = os.path.dirname(file_path)
output_file = os.path.join(output_dir, f"{base_name}_gpt.csv")

# Save the evaluated DataFrame to the interim data directory
df_evaluated.to_csv(output_file, index=False)
print(f"Saved evaluation file to {output_file}")